[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/memorization_generalization_in_diffusion_models_tests/blob/main/tp/TP_jacobien_base_adaptative.ipynb)

# Le débruiteur comme rétrécissement dans une base adaptative

On vérifie les propriétés du UNet sans biais sur lesquelles repose l'analyse de l'article (homogénéité, $f(y) = \nabla f(y)\,y$, symétrie du jacobien), puis on analyse son jacobien : valeurs propres (facteurs de rétrécissement), vecteurs propres (base adaptative), trace et erreur de débruitage, et ce qui change entre un réseau qui mémorise et un réseau qui généralise.

**Matériel** : tourne sur un ordinateur personnel (CPU) en quelques minutes ; sur Colab, exécuter d'abord la cellule d'installation (GPU facultatif).

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "memorization_generalization_in_diffusion_models_tests"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/code/',
            '/tp/tp_outils.py',
            '/tp/prepare_celeba80x80.py',
            '/denoisers/UNet/C_alpha4_beta4_line_80x80_blurredx2/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_10/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_99325/',
            '/results/img_align_celeba_80x80/']

import os, sys, subprocess
if "google.colab" in sys.modules:
    racine = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(racine, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), racine], check=True)
    subprocess.run(["git", "-C", racine, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", racine, "checkout", BRANCH], check=True)
    try:
        import pywt   # PyWavelets (générateur d'images C^alpha des auteurs) : absent de certaines images Colab
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "code", "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
os.makedirs(os.path.join(racine, "tp"), exist_ok=True)   # absent du clone partiel
os.chdir(os.path.join(racine, "tp"))
for d in ("code", "tp"):
    if os.path.join(racine, d) not in sys.path:
        sys.path.insert(0, os.path.join(racine, d))
print("Répertoire courant :", os.getcwd())

## 0. Réglages

Sur GPU, on calcule le jacobien complet ($6\,400 \times 6\,400$, soit 6 400 produits vecteur-jacobien). Sur
CPU, ce calcul prend plusieurs minutes : on calcule alors seulement les `K_VP` premiers vecteurs propres, sans
former la matrice (méthode de Lanczos). Les durées sont affichées.

In [ ]:
%matplotlib inline
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_jacobien_base_adaptative")   # figures nommées dans resultats/figures/

GRAINE = 0
K = 80
COMPLET = T.DEVICE.type == "cuda"   # jacobien complet sur GPU, Lanczos sur CPU
K_VP = 40
print("calcul sur", T.DEVICE, "; jacobien complet :", COMPLET)

## 1. Un UNet « sans biais »

On charge le débruiteur entraîné par les auteurs sur $10^5$ images $C^4$. Il renvoie le **résidu**
$r(y) = y - f(y)$ (le bruit estimé) ; l'image débruitée est $f(y) = y - r(y)$.

In [ ]:
reseau = T.charger_unet_auteurs("C_alpha4_beta4_line_80x80_blurredx2", 100000)
f = T.debruiteur_depuis_residu(reseau)
print("nombre de paramètres :", T.nombre_parametres(reseau))
noms = [nom for nom, _ in reseau.named_parameters()]
print("paramètres nommés « bias » :", [n for n in noms if "bias" in n])
print("types de paramètres :", sorted({n.split(".")[-1] for n in noms}))
print(reseau.encoder["0"])

### Homogénéité et $f(y) = \nabla f(y)\, y$

Sans constante additive et avec des ReLU, le réseau (en mode évaluation) vérifie $f(\lambda y) = \lambda f(y)$ pour
$\lambda > 0$. On le teste, ainsi que l'identité $f(y) = \nabla f(y)\,y$, calculée par un produit
jacobien-vecteur (`torch.func.jvp`), sans former le jacobien.

In [ ]:
from torch.func import jvp
x = T.images_calpha(4, 20, K, graine=GRAINE + 1)[9:10].to(T.DEVICE)
torch.manual_seed(GRAINE)
sigma = 100 / 255
y = x + sigma * torch.randn_like(x)
with torch.no_grad():
    f_y = f(y)
    for lam in [0.5, 2.0, 10.0, -1.0]:
        ecart = (f(lam * y) - lam * f_y).norm() / (lam * f_y).norm()
        print("λ = %5.1f : |f(λy) - λf(y)| / |λf(y)| = %.2e" % (lam, ecart))
Jy = jvp(f, (y,), (y,))[1].detach()
print("|f(y) - J(y) y| / |f(y)| = %.2e" % ((f_y - Jy).norm() / f_y.norm()))
reseau.train()
with torch.no_grad():
    ecart_train = (f(2 * y) - 2 * f(y)).norm() / (2 * f(y)).norm()
reseau.eval()
print("en mode entraînement (normalisation par les statistiques du lot) : %.2e" % ecart_train)

## 2. Valeurs et vecteurs propres du jacobien

Symétrie d'abord : on mesure $\|Jv - J^\top v\| / \|Jv\|$ pour des directions $v$ aléatoires. Puis on
diagonalise la partie symétrique $(J + J^\top)/2$ : valeurs propres $\lambda_k(y)$ (facteurs de
rétrécissement) et vecteurs propres $e_k(y)$. On trace aussi les coefficients $|\langle x, e_k(y)\rangle|$ de
l'image **propre** dans cette base.

In [ ]:
def analyse(f, x, y, titre, nom=None):
    import time
    print("asymétrie relative |Jv - J^T v| / |Jv| : %.2f" % T.asymetrie_jacobien(f, y))
    debut = time.time()
    if COMPLET:
        J = T.jacobien(f, y)
        valeurs, vecteurs, asym = T.decomposition_jacobien(J)
        print("jacobien complet : asymétrie |J - J^T| / |J| = %.2f, trace = %.1f" % (asym, valeurs.sum()))
    else:
        valeurs, vecteurs = T.vecteurs_propres_principaux(f, y, k=K_VP)
    print("%s : %.0f s" % ("jacobien complet" if COMPLET else "Lanczos, %d vecteurs propres" % K_VP,
                           time.time() - debut))
    valeurs, vecteurs = valeurs.cpu(), vecteurs.cpu()
    coefficients = (vecteurs.T @ x.flatten().cpu()).abs()
    with torch.no_grad():
        d_y = f(y)
    T.afficher_images(torch.cat([x.cpu(), y.cpu(), d_y.cpu()]), ["propre", "bruitée", "débruitée"], ncol=3,
                      taille=2.3, titre=titre, nom=nom and nom + "_images")
    fig, ax1 = plt.subplots(figsize=(6.5, 4))
    n = min(len(valeurs), 200)
    ax1.plot(range(1, n + 1), valeurs[:n], "r.", label="λ_k(y)")
    ax1.set_xlabel("k"); ax1.set_ylabel("λ_k(y)", color="r")
    ax2 = ax1.twinx()
    ax2.plot(range(1, n + 1), coefficients[:n], "b*", ms=4, label="|<x, e_k(y)>|")
    ax2.set_ylabel("|<x, e_k(y)>|", color="b")
    ax1.set_title(titre); ax1.grid(alpha=0.3)
    T.montrer(nom and nom + "_valeurs")
    print("10 premières valeurs propres :", np.round(valeurs[:10].numpy(), 3))
    part = (coefficients[:min(40, len(valeurs))] ** 2).sum() / (x.cpu() ** 2).sum()
    print("part de |x|² portée par les %d premiers vecteurs propres : %.3f" % (min(40, len(valeurs)), part))
    T.afficher_vecteurs(vecteurs[:, :21].T, K, ["λ_%d = %.2f" % (k + 1, valeurs[k]) for k in range(21)],
                        titre="vecteurs propres e_k(y)", nom=nom and nom + "_vecteurs")
    return valeurs, vecteurs

valeurs_c4, vecteurs_c4 = analyse(f, x, y, "C^4, σ = %.2f" % sigma, nom="calpha")

## 3. Trace du jacobien, dimension effective et erreur

La trace $\mathrm{tr}\,\nabla f(y) = \sum_k \lambda_k(y)$ mesure la « dimension » de l'espace conservé par le
débruiteur. On l'estime sans former $J$ (méthode de Hutchinson, `T.trace_jacobien`) pour plusieurs niveaux de
bruit, et on compare sur 20 images :

* l'erreur quadratique réelle $\|x - f(y)\|^2$ (qui demande l'image propre) ;
* l'estimation SURE $\|y - f(y)\|^2 + 2\sigma^2\,\mathrm{tr}\,\nabla f(y) - \sigma^2 d$ (qui n'utilise que $y$) ;
* $\sigma^2\,\mathrm{tr}\,\nabla f(y)$, égale en moyenne à l'erreur **si le débruiteur est optimal**.

In [ ]:
X20 = T.images_calpha(4, 20, K, graine=GRAINE + 2).to(T.DEVICE)
d = K * K
lignes = []
for sigma in [5 / 255, 10 / 255, 20 / 255, 50 / 255, 100 / 255, 200 / 255]:
    torch.manual_seed(GRAINE)
    vrai, sure, s2tr, traces = [], [], [], []
    for i in range(len(X20)):
        xi = X20[i:i + 1]
        yi = xi + sigma * torch.randn_like(xi)
        tr, _ = T.trace_jacobien(f, yi, n_sondes=16, graine=i)
        with torch.no_grad():
            fi = f(yi)
        vrai.append(((xi - fi) ** 2).sum().item())
        sure.append(((yi - fi) ** 2).sum().item() + 2 * sigma ** 2 * tr - sigma ** 2 * d)
        s2tr.append(sigma ** 2 * tr); traces.append(tr)
    lignes.append((sigma, np.mean(traces), np.mean(vrai), np.mean(sure), np.mean(s2tr)))
    print("σ = %.3f : tr J = %7.1f   erreur = %7.3f   SURE = %7.3f   σ² tr J = %7.3f" % lignes[-1])
lignes = np.array(lignes)
plt.figure(figsize=(6, 4))
plt.loglog(lignes[:, 0], lignes[:, 1], "o-")
plt.xlabel("σ"); plt.ylabel("tr ∇f(y) (moyenne sur 20 images)"); plt.grid(alpha=0.3)
plt.title("Dimension effective de l'espace conservé")
T.montrer("trace")

## 4. Mémorisation et généralisation vues par le jacobien (visages)

Les débruiteurs CelebA des auteurs entraînés sur $N = 10$ et $N = 10^5$ images (exactement 99 325 : la moitié
du jeu de données après retrait des doublons). Faute des images
d'entraînement (non distribuées), on prend comme image propre un échantillon **généré** et sauvegardé par
les auteurs : pour $N = 10$, c'est une copie d'une image d'entraînement (mémorisation) ; pour $N = 10^5$,
c'est un visage nouveau. On compare aussi les deux réseaux sur une même image.

In [ ]:
chemin = "../results/img_align_celeba_80x80/UNet_many_samples_%d.pt"
if os.path.exists(chemin % 10) and os.path.isdir("../denoisers/UNet/img_align_celeba_80x80"):
    visages = {}
    for N in [10, 100000]:
        ech = torch.load(chemin % N, map_location="cpu")["A"]
        valides = ech[(ech.amin(dim=(1, 2, 3)) > -0.05) & (ech.amax(dim=(1, 2, 3)) < 1.05)]
        print("N = %6d : %d échantillons sauvegardés, dont %d à valeurs dans [-0.05, 1.05]"
              % (N, len(ech), len(valides)))
        visages[N] = valides
    sigma = 40 / 255
    for N_reseau, N_image in [(100000, 100000), (10, 10), (10, 100000)]:
        r = T.charger_unet_auteurs("img_align_celeba_80x80", 99325 if N_reseau == 100000 else N_reseau)
        xv = visages[N_image][3:4].to(T.DEVICE)
        torch.manual_seed(GRAINE)
        yv = xv + sigma * torch.randn_like(xv)
        print("\n=== réseau N = %d, image générée par le réseau N = %d" % (N_reseau, N_image))
        analyse(T.debruiteur_depuis_residu(r), xv, yv, "réseau N = %d, image du réseau N = %d" % (N_reseau, N_image),
                nom="visage_reseau%d_image%d" % (N_reseau, N_image))
else:
    print("Poids ou échantillons CelebA absents : section ignorée (voir SPARSE_DIRS).")